# Actividad Calificable · Corte 2
## Modelo, consulta y limpieza de datos
### 🌱 Caso: Gestión del Riego Agroindustrial basada en Datos

```yaml
# CONFIG
FULL_NAME: Isabella Ramos Betancourt
GITHUB_USER: Isarb-21
```

**Isabella Ramos Betancourt** · Facultad de Ingeniería, CORHUILA  
Ciencia de Datos (Cód. 69109) [Pénsum 40D, Grupo 1] · Profesor: Jesús Ariel Gonzáles Bonilla · Octubre de 2026

**Cómo usar este cuaderno:** ejecútelo con *Restart & Run All* desde la carpeta de la semana (junto a `lectura_sensor_raw.csv`) y guárdelo con las salidas visibles. Todas las cifras de este cuaderno salen de la ejecución, no están escritas a mano.

**Contenido:** 1) carga y diagnóstico, 2) limpieza, 3) verificación antes/después, 4) base relacional y dos consultas con hallazgo.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

candidatos = [Path("lectura_sensor_raw.csv"), Path("09-week/lectura_sensor_raw.csv")]
RAW = next((p for p in candidatos if p.exists()), None)
if RAW is None:
    raise FileNotFoundError("No se encontró lectura_sensor_raw.csv junto al cuaderno ni en 09-week/")
CLEAN = RAW.with_name("lectura_sensor_clean.csv")
print("Dataset bruto:", RAW)

Dataset bruto: lectura_sensor_raw.csv


## 1. Carga e inspección inicial (ANTES)

In [2]:
raw = pd.read_csv(RAW)

print("Forma:", raw.shape)
print("\nTipos de datos:")
print(raw.dtypes)
print("\nNulos por columna:")
print(raw.isnull().sum())
print("\nVariantes de id_lote:", sorted(raw["id_lote"].astype(str).unique()))
raw.head(8)

Forma: (75, 5)

Tipos de datos:
id_lectura             int64
id_lote                  str
fecha_hora               str
humedad_suelo_pct    float64
temperatura_c            str
dtype: object

Nulos por columna:
id_lectura           0
id_lote              0
fecha_hora           0
humedad_suelo_pct    2
temperatura_c        1
dtype: int64

Variantes de id_lote: ['LOT 001', 'LOT-001', 'LOT-002', 'LOT-003', 'LOT001', 'lot002', 'lote-003']


,id_lectura,id_lote,fecha_hora,humedad_suelo_pct,temperatura_c
0,1,LOT-001,2026-09-01 00:00,46.2,21.8
1,2,LOT 001,2026-09-01 01:00,46.6,24.3
2,3,LOT-001,01/09/2026 02:00,44.4,21.6
3,4,LOT-001,2026-09-01 03:00,150.0,23.2
4,5,lot002,2026-09-01 04:00,43.8,22.8
5,6,LOT-001,2026-09-01 05:00,43.8,ERROR_SENSOR
6,7,LOT-001,2026-09-01 06:00,NaN,19.1
7,8,LOT-001,2026-09-01 07:00,40.7,21.2


In [3]:
# Métricas "ANTES" calculadas sobre el archivo bruto (se usan en la tabla comparativa)
hum_raw  = pd.to_numeric(raw["humedad_suelo_pct"], errors="coerce")
temp_raw = pd.to_numeric(raw["temperatura_c"], errors="coerce")

antes = {
    "filas": len(raw),
    "duplicados_exactos": int(raw.duplicated().sum()),
    "variantes_id_lote": int(raw["id_lote"].astype(str).nunique()),
    "dtype_fecha_hora": str(raw["fecha_hora"].dtype),
    "dtype_temperatura_c": str(raw["temperatura_c"].dtype),
    "nulos_humedad": int(hum_raw.isna().sum()),
    "nulos_temperatura_originales": int(raw["temperatura_c"].isna().sum()),
    "temperatura_texto_invalido": int((temp_raw.isna() & raw["temperatura_c"].notna()).sum()),
    "outliers_humedad": int(((hum_raw > 100) | (hum_raw < 0)).sum()),
    "outliers_temperatura": int((temp_raw.notna() & ((temp_raw > 50) | (temp_raw < 0))).sum()),
}
pd.Series(antes, name="ANTES").to_frame()

,ANTES
filas,75
duplicados_exactos,3
variantes_id_lote,7
dtype_fecha_hora,str
dtype_temperatura_c,str
nulos_humedad,2
nulos_temperatura_originales,1
temperatura_texto_invalido,2
outliers_humedad,2
outliers_temperatura,0


## 2. Limpieza paso a paso

Reglas: `id_lote` canónico `LOT-XXX`; `fecha_hora` a `datetime64` (se asume formato día/mes/año cuando hay ambigüedad); `temperatura_c` numérica; humedad válida en [0, 100] %; temperatura válida en [0, 50] °C; faltantes imputados con la **mediana por lote**.

In [4]:
df = raw.copy()

# PASO 1: normalizar id_lote (extrae el número y reconstruye el código canónico)
num = df["id_lote"].astype(str).str.extract(r"(\d+)")[0]
assert num.notna().all(), "Hay id_lote sin número: revisar manualmente"
df["id_lote"] = num.astype(int).map("LOT-{:03d}".format)
print("[Paso 1] Lotes canónicos:", sorted(df["id_lote"].unique()))
print("[Paso 1] Duplicados tras normalizar id_lote:", int(df.duplicated().sum()))

# PASO 2: fecha_hora a datetime
# Las fechas ISO (AAAA-MM-DD) se leen aparte: dayfirst=True las confundiría (2026-09-01 -> 9 de enero)
txt = df["fecha_hora"].astype(str).str.strip()
es_iso = txt.str.match(r"^\d{4}[-/]\d{2}[-/]\d{2}")   # AAAA-MM-DD o AAAA/MM/DD
df["fecha_hora"] = pd.to_datetime(txt.where(es_iso).str.replace("/", "-"), format="ISO8601").fillna(
    pd.to_datetime(txt.where(~es_iso), format="mixed", dayfirst=True))
print("[Paso 2] Tipo de fecha_hora:", df["fecha_hora"].dtype)
print("[Paso 2] Rango de fechas:", df["fecha_hora"].min(), "->", df["fecha_hora"].max())
assert df["fecha_hora"].dt.strftime("%Y-%m").nunique() == 1, "Fechas en meses distintos: revisar el parseo"
dup_tras_normalizar = int(df.duplicated().sum())
print("[Paso 2] Duplicados tras normalizar id_lote y fecha_hora:", dup_tras_normalizar)

# PASO 3: tipos numéricos (el texto inválido pasa a NaN)
df["temperatura_c"] = pd.to_numeric(df["temperatura_c"], errors="coerce")
df["humedad_suelo_pct"] = pd.to_numeric(df["humedad_suelo_pct"], errors="coerce")
print("[Paso 3] Nulos de temperatura tras coerción:", int(df["temperatura_c"].isna().sum()))

# PASO 4: eliminar duplicados (ya con claves y fechas normalizadas)
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"[Paso 4] Duplicados eliminados: {n0 - len(df)} -> filas: {len(df)}")

# PASO 5: valores fuera de rango -> NaN
m_hum = (df["humedad_suelo_pct"] > 100) | (df["humedad_suelo_pct"] < 0)
m_tmp = df["temperatura_c"].notna() & ((df["temperatura_c"] > 50) | (df["temperatura_c"] < 0))
print("[Paso 5] Humedad fuera de [0,100]:", int(m_hum.sum()), "| Temperatura fuera de [0,50]:", int(m_tmp.sum()))
df.loc[m_hum, "humedad_suelo_pct"] = np.nan
df.loc[m_tmp, "temperatura_c"] = np.nan

# PASO 6: imputación con la mediana por lote
nulos_a_imputar = int(df[["humedad_suelo_pct", "temperatura_c"]].isna().sum().sum())
print("[Paso 6] Valores a imputar:", nulos_a_imputar,
      "| humedad:", int(df["humedad_suelo_pct"].isna().sum()),
      "| temperatura:", int(df["temperatura_c"].isna().sum()))
for col in ["humedad_suelo_pct", "temperatura_c"]:
    df[col] = df.groupby("id_lote")[col].transform(lambda s: s.fillna(s.median()))

# PASO 7: exportar
df.to_csv(CLEAN, index=False)
print("[Paso 7] Guardado:", CLEAN)

[Paso 1] Lotes canónicos: ['LOT-001', 'LOT-002', 'LOT-003']
[Paso 1] Duplicados tras normalizar id_lote: 3
[Paso 2] Tipo de fecha_hora: datetime64[us]
[Paso 2] Rango de fechas: 2026-09-01 00:00:00 -> 2026-09-01 23:00:00
[Paso 2] Duplicados tras normalizar id_lote y fecha_hora: 3
[Paso 3] Nulos de temperatura tras coerción: 3
[Paso 4] Duplicados eliminados: 3 -> filas: 72
[Paso 5] Humedad fuera de [0,100]: 2 | Temperatura fuera de [0,50]: 0
[Paso 6] Valores a imputar: 7 | humedad: 4 | temperatura: 3
[Paso 7] Guardado: lectura_sensor_clean.csv


## 3. Verificación y reporte ANTES vs. DESPUÉS

In [5]:
print("Forma final:", df.shape)
print("\nTipos finales:")
print(df.dtypes)
print("\nNulos restantes:")
print(df.isnull().sum())
print("\nLotes:", sorted(df["id_lote"].unique()))
print("Duplicados restantes:", int(df.duplicated().sum()))
print("Rango humedad_suelo_pct:", df["humedad_suelo_pct"].min(), "-", df["humedad_suelo_pct"].max())
print("Rango temperatura_c    :", df["temperatura_c"].min(), "-", df["temperatura_c"].max())

assert df.isnull().sum().sum() == 0
assert df["humedad_suelo_pct"].between(0, 100).all()
assert df["temperatura_c"].between(0, 50).all()
assert df.duplicated().sum() == 0

Forma final: (72, 5)

Tipos finales:
id_lectura                    int64
id_lote                         str
fecha_hora           datetime64[us]
humedad_suelo_pct           float64
temperatura_c               float64
dtype: object

Nulos restantes:
id_lectura           0
id_lote              0
fecha_hora           0
humedad_suelo_pct    0
temperatura_c        0
dtype: int64

Lotes: ['LOT-001', 'LOT-002', 'LOT-003']
Duplicados restantes: 0
Rango humedad_suelo_pct: 28.1 - 55.8
Rango temperatura_c    : 19.1 - 27.6


In [6]:
despues = {
    "filas": len(df),
    "duplicados_exactos": int(df.duplicated().sum()),
    "variantes_id_lote": int(df["id_lote"].nunique()),
    "dtype_fecha_hora": str(df["fecha_hora"].dtype),
    "dtype_temperatura_c": str(df["temperatura_c"].dtype),
    "nulos_humedad": int(df["humedad_suelo_pct"].isna().sum()),
    "nulos_temperatura_originales": int(df["temperatura_c"].isna().sum()),
    "temperatura_texto_invalido": 0,
    "outliers_humedad": int(((df["humedad_suelo_pct"] > 100) | (df["humedad_suelo_pct"] < 0)).sum()),
    "outliers_temperatura": int(((df["temperatura_c"] > 50) | (df["temperatura_c"] < 0)).sum()),
}
comparativo = pd.DataFrame({"ANTES": antes, "DESPUÉS": despues})
print("Valores imputados en total:", nulos_a_imputar)
print("Duplicados detectados tras normalizar claves y fechas (antes de eliminar):", dup_tras_normalizar)
comparativo

Valores imputados en total: 7
Duplicados detectados tras normalizar claves y fechas (antes de eliminar): 3


,ANTES,DESPUÉS
filas,75,72
duplicados_exactos,3,0
variantes_id_lote,7,3
dtype_fecha_hora,str,datetime64[us]
dtype_temperatura_c,str,float64
nulos_humedad,2,0
nulos_temperatura_originales,1,0
temperatura_texto_invalido,2,0
outliers_humedad,2,0
outliers_temperatura,0,0


## 4. Base relacional y consultas

Se crean las 5 tablas del ERD en SQLite (en memoria) con claves foráneas activas. `lectura_sensor` se carga desde el dataset **limpio**. Los datos de lotes, técnicos, riegos y temporada son **datos de ejemplo** del caso.

In [7]:
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")

conn.executescript('''
CREATE TABLE lote (
    id_lote TEXT PRIMARY KEY, nombre TEXT NOT NULL, area_ha REAL,
    tipo_suelo TEXT, cultivo TEXT
);
CREATE TABLE tecnico (
    id_tecnico TEXT PRIMARY KEY, nombre TEXT NOT NULL, especialidad TEXT, telefono TEXT
);
CREATE TABLE registro_riego (
    id_riego INTEGER PRIMARY KEY,
    id_lote TEXT NOT NULL REFERENCES lote(id_lote),
    id_tecnico TEXT NOT NULL REFERENCES tecnico(id_tecnico),
    fecha TEXT NOT NULL, volumen_agua_m3 REAL NOT NULL, duracion_min INTEGER
);
CREATE TABLE temporada_cosecha (
    id_temporada INTEGER PRIMARY KEY,
    id_lote TEXT NOT NULL REFERENCES lote(id_lote),
    fecha_inicio TEXT NOT NULL, fecha_fin TEXT NOT NULL, rendimiento_ton REAL
);
''')

conn.executemany("INSERT INTO lote VALUES (?,?,?,?,?)", [
    ("LOT-001", "Lote Norte - Maíz",  5.2, "Franco-arcilloso", "Maíz dulce"),
    ("LOT-002", "Lote Sur - Tomate",  3.8, "Arenoso",          "Tomate chonto"),
    ("LOT-003", "Lote Este - Café",   6.0, "Franco-limoso",    "Café castillo"),
])
conn.executemany("INSERT INTO tecnico VALUES (?,?,?,?)", [
    ("TEC-01", "Carlos Mendoza", "Riego por goteo",      "3101234567"),
    ("TEC-02", "María Ospina",   "Monitoreo de humedad", "3209876543"),
])
conn.executemany("INSERT INTO registro_riego VALUES (?,?,?,?,?,?)", [
    (1, "LOT-001", "TEC-01", "2026-09-01", 3.4, 45),
    (2, "LOT-001", "TEC-02", "2026-09-03", 2.8, 35),
    (3, "LOT-001", "TEC-01", "2026-09-05", 4.1, 55),
    (4, "LOT-002", "TEC-02", "2026-09-01", 5.0, 60),
    (5, "LOT-002", "TEC-01", "2026-09-04", 4.8, 58),
    (6, "LOT-003", "TEC-02", "2026-09-02", 1.9, 30),
    (7, "LOT-003", "TEC-01", "2026-09-06", 2.2, 35),
])
conn.executemany("INSERT INTO temporada_cosecha VALUES (?,?,?,?,?)", [
    (1, "LOT-001", "2026-03-01", "2026-09-30", 28.5),
    (2, "LOT-002", "2026-03-01", "2026-09-30", 42.0),
    (3, "LOT-003", "2026-03-01", "2026-09-30", 18.2),
])

sensores = pd.read_csv(CLEAN)
sensores["fecha_hora"] = pd.to_datetime(sensores["fecha_hora"]).dt.strftime("%Y-%m-%d %H:%M:%S")
sensores.to_sql("lectura_sensor", conn, index=False, if_exists="replace")
conn.commit()

pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)

,name
0,lectura_sensor
1,lote
2,registro_riego
3,tecnico
4,temporada_cosecha


### Consulta 1 · Riegos del Lote Norte por encima de su propio promedio (filtro `WHERE`)

**Pregunta 1:** ¿En qué fechas `LOT-001` recibió más agua que el promedio de sus propios riegos?

In [8]:
sql_1 = '''
SELECT fecha, volumen_agua_m3, duracion_min,
       ROUND((volumen_agua_m3 / (SELECT AVG(volumen_agua_m3) FROM registro_riego WHERE id_lote = 'LOT-001') - 1) * 100, 1)
           AS pct_sobre_promedio
FROM registro_riego
WHERE id_lote = 'LOT-001'
  AND volumen_agua_m3 > (SELECT AVG(volumen_agua_m3) FROM registro_riego WHERE id_lote = 'LOT-001')
ORDER BY fecha;
'''
promedio = pd.read_sql_query(
    "SELECT AVG(volumen_agua_m3) AS p FROM registro_riego WHERE id_lote='LOT-001'", conn)["p"][0]
print(f"Volumen promedio de riego en LOT-001: {promedio:.2f} m³")
c1 = pd.read_sql_query(sql_1, conn)
c1

Volumen promedio de riego en LOT-001: 3.43 m³


,fecha,volumen_agua_m3,duracion_min,pct_sobre_promedio
0,2026-09-05,4.1,55,19.4


In [9]:
# ¿Los sensores tienen lecturas en las fechas de esos riegos?
hum_dia = pd.read_sql_query('''
    SELECT DATE(fecha_hora) AS fecha, COUNT(*) AS lecturas, ROUND(AVG(humedad_suelo_pct), 1) AS humedad_promedio_pct
    FROM lectura_sensor WHERE id_lote = 'LOT-001' GROUP BY DATE(fecha_hora) ORDER BY fecha;
''', conn)
print("Cobertura de sensores en LOT-001:")
print(hum_dia.to_string(index=False))

for _, ev in c1.iterrows():
    fila = hum_dia[hum_dia["fecha"] == ev["fecha"]]
    if fila.empty:
        print(f"\n{ev['fecha']}: riego de {ev['volumen_agua_m3']} m³ (+{ev['pct_sobre_promedio']} % sobre el promedio). "
              "NO hay lecturas de sensor ese día: no se puede concluir sobre-irrigación; queda como hipótesis.")
    else:
        h = fila["humedad_promedio_pct"].iloc[0]
        print(f"\n{ev['fecha']}: riego de {ev['volumen_agua_m3']} m³ (+{ev['pct_sobre_promedio']} % sobre el promedio). "
              f"Humedad promedio del suelo ese día: {h} %.")

Cobertura de sensores en LOT-001:
     fecha  lecturas  humedad_promedio_pct
2026-09-01        24                  43.9

2026-09-05: riego de 4.1 m³ (+19.4 % sobre el promedio). NO hay lecturas de sensor ese día: no se puede concluir sobre-irrigación; queda como hipótesis.


**Hallazgo (Consulta 1):** el filtro devuelve únicamente los riegos estrictamente superiores al promedio de `LOT-001`. El porcentaje se calcula contra el **promedio** del lote (no contra el umbral de 3.0 m³ de versiones anteriores). La celda siguiente indica, con datos, si los sensores cubren esas fechas; solo en ese caso se interpreta la humedad. Si no las cubren, la sobre-irrigación es una **hipótesis a verificar** con lecturas de ese día, no una conclusión.

### Consulta 2 · Consumo y eficiencia hídrica por lote y temporada (`JOIN` + `GROUP BY`)

**Pregunta 2:** ¿Cuánta agua recibió cada lote **dentro de su temporada** y qué rendimiento se obtuvo por m³ y por hectárea?  
El agua se agrega primero por temporada (CTE) y luego se une con el rendimiento, para que un lote con varias temporadas no duplique filas ni infle el `SUM`.

In [10]:
sql_2 = '''
WITH agua AS (
    SELECT tc.id_temporada, r.id_lote,
           SUM(r.volumen_agua_m3) AS agua_total_m3,
           COUNT(*)               AS num_riegos
    FROM temporada_cosecha tc
    JOIN registro_riego r
      ON r.id_lote = tc.id_lote
     AND r.fecha BETWEEN tc.fecha_inicio AND tc.fecha_fin
    GROUP BY tc.id_temporada, r.id_lote
)
SELECT l.nombre AS lote, l.cultivo, l.area_ha, a.num_riegos,
       ROUND(a.agua_total_m3, 1)                       AS agua_total_m3,
       tc.rendimiento_ton,
       ROUND(tc.rendimiento_ton / a.agua_total_m3, 3)  AS ton_por_m3,
       ROUND(a.agua_total_m3 / l.area_ha, 2)           AS m3_por_ha,
       ROUND(tc.rendimiento_ton / l.area_ha, 2)        AS ton_por_ha
FROM agua a
JOIN temporada_cosecha tc ON tc.id_temporada = a.id_temporada
JOIN lote l               ON l.id_lote = a.id_lote
ORDER BY ton_por_m3 DESC;
'''
c2 = pd.read_sql_query(sql_2, conn)
c2

,lote,cultivo,area_ha,num_riegos,agua_total_m3,rendimiento_ton,ton_por_m3,m3_por_ha,ton_por_ha
0,Lote Este - Café,Café castillo,6.0,2,4.1,18.2,4.439,0.68,3.03
1,Lote Sur - Tomate,Tomate chonto,3.8,2,9.8,42.0,4.286,2.58,11.05
2,Lote Norte - Maíz,Maíz dulce,5.2,3,10.3,28.5,2.767,1.98,5.48


In [11]:
# Réplica en pandas (groupby) y comprobación de equivalencia con el SQL
lote_df = pd.read_sql_query("SELECT * FROM lote", conn)
riego_df = pd.read_sql_query("SELECT * FROM registro_riego", conn)
temp_df = pd.read_sql_query("SELECT * FROM temporada_cosecha", conn)

m = riego_df.merge(temp_df, on="id_lote")
m = m[(m["fecha"] >= m["fecha_inicio"]) & (m["fecha"] <= m["fecha_fin"])]
agua = (m.groupby(["id_temporada", "id_lote", "rendimiento_ton"], as_index=False)
          .agg(agua_total_m3=("volumen_agua_m3", "sum"), num_riegos=("id_riego", "count")))
res = agua.merge(lote_df[["id_lote", "nombre", "cultivo", "area_ha"]], on="id_lote")
res["ton_por_m3"] = (res["rendimiento_ton"] / res["agua_total_m3"]).round(3)
res["m3_por_ha"] = (res["agua_total_m3"] / res["area_ha"]).round(2)
res["ton_por_ha"] = (res["rendimiento_ton"] / res["area_ha"]).round(2)
res["agua_total_m3"] = res["agua_total_m3"].round(1)
res = res.sort_values("ton_por_m3", ascending=False).reset_index(drop=True)

cols = ["cultivo", "num_riegos", "agua_total_m3", "rendimiento_ton", "ton_por_m3", "m3_por_ha", "ton_por_ha"]
pd.testing.assert_frame_equal(res[cols], c2[cols], check_dtype=False)
print("✓ pandas y SQL producen el mismo resultado")
res[["nombre"] + cols]

✓ pandas y SQL producen el mismo resultado


,nombre,cultivo,num_riegos,agua_total_m3,rendimiento_ton,ton_por_m3,m3_por_ha,ton_por_ha
0,Lote Este - Café,Café castillo,2,4.1,18.2,4.439,0.68,3.03
1,Lote Sur - Tomate,Tomate chonto,2,9.8,42.0,4.286,2.58,11.05
2,Lote Norte - Maíz,Maíz dulce,3,10.3,28.5,2.767,1.98,5.48


**Hallazgo (Consulta 2):** `ton_por_m3` ordena los lotes por rendimiento por metro cúbico, pero **no es una comparación justa entre cultivos distintos** (maíz, tomate y café tienen requerimientos hídricos y rendimientos por hectárea muy diferentes), y las áreas de los lotes también difieren; por eso se añaden `m3_por_ha` y `ton_por_ha`.

Limitaciones que deben quedar declaradas:
- El rendimiento corresponde a **toda la temporada**, mientras que los riegos disponibles cubren solo unos días de septiembre; la razón ton/m³ es **indicativa**, no una medida de eficiencia real.
- Los datos de lotes, riegos y temporada son de ejemplo.
- Una decisión de inversión (por ejemplo, tecnificar el riego del maíz) requeriría varias temporadas del mismo cultivo y el registro completo de riegos.

## 5. Conclusiones

1. **Limpieza verificada:** la tabla comparativa y las aserciones de la sección 3 demuestran el estado final del dataset (sin nulos, sin duplicados, tipos y rangos válidos); los conteos provienen de la ejecución.
2. **Consulta 1:** el criterio "sobre el promedio del propio lote" es explícito y reproducible; la interpretación agronómica depende de que existan lecturas de sensor en la fecha del riego.
3. **Consulta 2:** SQL y pandas coinciden; el indicador ton/m³ se complementa con métricas por hectárea y se presenta con sus limitaciones.

**Referencias:** FAO (2026). *Water data and resource assessment*. https://www.fao.org/land-water/water/water-data-and-resource-assessment · IBM (2024). *What is data quality?* https://www.ibm.com/think/topics/data-quality · McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly. · Silberschatz, Korth & Sudarshan (2020). *Database System Concepts* (7th ed.). McGraw-Hill.